# PASAD and EPASAD Anomaly Detection
This notebook implements **PASAD** (Process-Aware Stealthy Attack Detection) and its variant **EPASAD**. 

### Background
**Singular Spectrum Analysis (SSA)** is a technique used to decompose time-series data into a sum of components (trend, oscillatory behavior, and noise). 
1. **Trajectory Matrix**: We construct a Hankel matrix from a single sensor's time series using a sliding window of length $L$.
2. **Subspace Projection**: Using SVD on the Hankel matrix, we identify the top $r$ singular vectors ($U_r$), which represent the "normal signal subspace".
3. **PASAD vs EPASAD**: 
   - During training, we project normal windows onto $U_r$ and compute a centroid.
   - **PASAD** computes the squared Euclidean distance between a new test window's projection and the centroid.
   - **EPASAD** computes the Mahalanobis distance, using the covariance of the training projections, allowing for elliptical boundaries rather than strictly spherical ones.


In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, f1_score, roc_auc_score
import joblib

# Parameters
STAGE_NUM = 1 # 1 to 6
DEFAULT_L = 50 # Default window length per tag (tune per tag using autocorrelation)
DEFAULT_r = 5  # Default subspace dimension

# Dictionary overriding L and r per specific tag name (if dynamics differ)
L_dict = {}
r_dict = {}

STRIDE = 1 # Lag/stride for the sliding window used during scoring
THRESHOLD_PERCENTILE = 99.9 # For deriving the anomaly threshold from normal data fused score


### Preprocessing
PASAD is designed for **continuous time-series**. We explicitly exclude binary actuator state columns (like MV and P series). We normalize each continuous tag independently using a `StandardScaler` fitted **only** on that tag's normal data. Scaling is crucial because PASAD relies on subspace geometry, which is heavily distorted by varying signal magnitudes.

In [ ]:
def load_and_preprocess(stage_num):
    normal_path = f'dataset/normal_stage{stage_num}.csv'
    attack_path = f'dataset/attack_stage{stage_num}.csv'
    
    df_normal = pd.read_csv(normal_path)
    df_attack = pd.read_csv(attack_path)
    
    # Parse Timestamp and Sort
    df_normal['Timestamp'] = pd.to_datetime(df_normal['Timestamp'], format='%d/%m/%Y %I:%M:%S %p', errors='coerce')
    df_normal = df_normal.sort_values('Timestamp').reset_index(drop=True)
    df_normal.ffill(inplace=True)
    
    df_attack['Timestamp'] = pd.to_datetime(df_attack['Timestamp'], format='%d/%m/%Y %I:%M:%S %p', errors='coerce')
    df_attack = df_attack.sort_values('Timestamp').reset_index(drop=True)
    df_attack.ffill(inplace=True)
    
    # Identify continuous columns vs discrete
    # We'll drop Timestamp, attack, and any column with MV or P (which are typically discrete states)
    exclude_cols = ['Timestamp', 'attack']
    continuous_cols = [c for c in df_normal.columns if c not in exclude_cols and not (c.startswith('MV') or c.startswith('P'))]
    
    return df_normal, df_attack, continuous_cols

def normalize_data(df_normal, df_attack, columns):
    scalers = {}
    norm_scaled = df_normal.copy()
    att_scaled = df_attack.copy()
    
    for col in columns:
        scaler = StandardScaler()
        # FIT ONLY on normal data
        norm_scaled[col] = scaler.fit_transform(df_normal[[col]])
        att_scaled[col] = scaler.transform(df_attack[[col]])
        scalers[col] = scaler
        
    return norm_scaled, att_scaled, scalers


### PASAD Training and Scoring
- **`build_hankel`**: Creates overlapping windows.
- **`train_pasad`**: Computes the Hankel matrix, applies SVD to find $U_r$, and calculates the training centroid (and covariance for EPASAD).
- **`score_pasad`**: Slides a window over the test signal, projects onto $U_r$, and measures distance to the centroid.

In [ ]:
from scipy.linalg import svd

def build_hankel(ts, L):
    N = len(ts)
    if N < L:
        return np.empty((L, 0))
    K = N - L + 1
    # Create the Hankel matrix: L rows, K columns
    X = np.column_stack([ts[i:i+L] for i in range(K)])
    return X

def train_pasad(ts_norm, L, r):
    # 1. Build Hankel Matrix
    X = build_hankel(ts_norm, L)
    K = X.shape[1]
    
    # 2. SVD
    # X = U * S * V^T
    # We only need the top r left singular vectors
    U, S, Vt = svd(X, full_matrices=False)
    U_r = U[:, :r] # Shape: (L, r)
    
    # 3. Project normal training windows onto U_r
    # Projections = U_r^T * X -> Shape: (r, K)
    projections = U_r.T @ X
    
    # 4. Compute centroid
    centroid = np.mean(projections, axis=1) # Shape: (r,)
    
    # 5. Compute covariance for EPASAD
    # Shape: (r, r)
    cov = np.cov(projections)
    # Add small epsilon to diagonal for numerical stability in case of singularity
    cov += np.eye(r) * 1e-6
    cov_inv = np.linalg.inv(cov)
    
    return U_r, centroid, cov, cov_inv

def score_pasad(ts_test, U_r, centroid, cov_inv, L, stride=1):
    N = len(ts_test)
    scores_pasad = []
    scores_epasad = []
    
    # Indices for the start of each sliding window
    starts = range(0, N - L + 1, stride)
    
    for i in starts:
        window = ts_test[i:i+L]
        
        # Project window onto U_r
        # window is (L,), U_r.T is (r, L) -> proj is (r,)
        proj = U_r.T @ window
        
        # PASAD Score (Squared Euclidean distance)
        diff = proj - centroid
        dist_pasad = np.sum(diff**2)
        scores_pasad.append(dist_pasad)
        
        # EPASAD Score (Mahalanobis distance)
        dist_epasad = diff.T @ cov_inv @ diff
        scores_epasad.append(dist_epasad)
        
    return np.array(scores_pasad), np.array(scores_epasad), list(starts)


### Fusion and Thresholding
We min-max normalize each tag's anomaly score (using its own normal-window score distribution) to make them comparable. Then we fuse them by taking the `max` or `mean` across tags per timestep.

In [ ]:
def fuse_tag_scores(scores_dict, method='max'):
    '''
    scores_dict: { tag_name: score_array }
    method: 'max' or 'mean'
    '''
    df_scores = pd.DataFrame(scores_dict)
    
    if method == 'max':
        return df_scores.max(axis=1).values
    elif method == 'mean':
        return df_scores.mean(axis=1).values
    else:
        raise ValueError("Method must be 'max' or 'mean'")

def compute_threshold(normal_fused_scores, method='mean_std'):
    if method == 'mean_std':
        # e.g., mean + 3*std
        return np.mean(normal_fused_scores) + 3 * np.std(normal_fused_scores)
    elif method == 'percentile':
        return np.percentile(normal_fused_scores, THRESHOLD_PERCENTILE)
    else:
        return np.percentile(normal_fused_scores, 99.9)


### Evaluation Metrics
We compute standard classification metrics. We also track which tags contributed the most to the overall detection.

In [ ]:
def evaluate(y_true, y_score, threshold, tag_scores, tag_thresholds):
    y_pred = (y_score > threshold).astype(int)
    
    cm = confusion_matrix(y_true, y_pred)
    # cm: [[TN, FP], [FN, TP]]
    TN, FP, FN, TP = cm.ravel()
    
    fpr = FP / (FP + TN) if (FP + TN) > 0 else 0
    fnr = FN / (FN + TP) if (FN + TP) > 0 else 0
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    try:
        roc_auc = roc_auc_score(y_true, y_score)
    except:
        roc_auc = 0.5 # if only one class present
        
    print(f"Confusion Matrix:\n{cm}")
    print(f"Precision: {precision:.4f}, Recall/TPR: {recall:.4f}, F1: {f1:.4f}")
    print(f"FPR: {fpr:.4f}, FNR: {fnr:.4f}, ROC-AUC: {roc_auc:.4f}")
    
    # Tag contribution during True Positives
    tp_indices = np.where((y_true == 1) & (y_pred == 1))[0]
    
    tag_contributions = {}
    for tag in tag_scores.keys():
        # How many times did this tag exceed its own threshold during TP windows?
        exceed_count = np.sum(tag_scores[tag][tp_indices] > tag_thresholds[tag])
        tag_contributions[tag] = exceed_count
        
    # Plot Tag Contributions
    plt.figure(figsize=(12, 6))
    tags = list(tag_contributions.keys())
    counts = list(tag_contributions.values())
    plt.bar(tags, counts)
    plt.xticks(rotation=90)
    plt.title("Tag Threshold Exceedances during True Positive Windows")
    plt.ylabel("Exceedance Count")
    plt.tight_layout()
    plt.show()
    
    return {'Precision': precision, 'Recall': recall, 'F1': f1, 'FPR': fpr, 'FNR': fnr, 'ROC-AUC': roc_auc}


### Main Pipeline
Runs the entire pipeline (training on normal, testing on normal to find threshold, testing on attack) for all stages.

In [ ]:
results = []
models_dir = 'models'
os.makedirs(models_dir, exist_ok=True)

# Try running on stages 1 to 6
for stg in range(1, 7):
    print(f"\n{'='*40}\nProcessing STAGE {stg}\n{'='*40}")
    try:
        df_norm, df_att, cont_cols = load_and_preprocess(stg)
    except FileNotFoundError:
        print(f"Data for stage {stg} not found. Skipping.")
        continue
        
    df_norm_scaled, df_att_scaled, scalers = normalize_data(df_norm, df_att, cont_cols)
    
    # Store tag-level data
    tag_models = {}
    
    # Normal scoring to find min/max for normalization and final threshold
    norm_pasad_raw = {}
    norm_epasad_raw = {}
    
    # Attack scoring
    att_pasad_raw = {}
    att_epasad_raw = {}
    
    latency_measurements = []
    
    # Per-tag Training & Scoring
    print(f"Training and Scoring per tag ({len(cont_cols)} tags)...")
    for tag in cont_cols:
        L = L_dict.get(tag, DEFAULT_L)
        r = r_dict.get(tag, DEFAULT_r)
        
        ts_norm = df_norm_scaled[tag].values
        ts_att = df_att_scaled[tag].values
        
        # 1. Train
        U_r, centroid, cov, cov_inv = train_pasad(ts_norm, L, r)
        
        # 2. Score Normal (to compute normalization bounds and threshold)
        scores_pasad_norm, scores_epasad_norm, _ = score_pasad(ts_norm, U_r, centroid, cov_inv, L, STRIDE)
        
        # 3. Score Attack
        start_time = time.time()
        scores_pasad_att, scores_epasad_att, att_starts = score_pasad(ts_att, U_r, centroid, cov_inv, L, STRIDE)
        end_time = time.time()
        
        # Average Latency per window (in milliseconds)
        latency_measurements.append((end_time - start_time) / len(att_starts) * 1000)
        
        norm_pasad_raw[tag] = scores_pasad_norm
        norm_epasad_raw[tag] = scores_epasad_norm
        att_pasad_raw[tag] = scores_pasad_att
        att_epasad_raw[tag] = scores_epasad_att
        
        tag_models[tag] = {
            'U_r': U_r, 'centroid': centroid, 'cov': cov, 'scaler': scalers[tag]
        }
    
    print(f"Mean inference latency per tag window: {np.mean(latency_measurements):.4f} ms")
    print(f"p95 inference latency per tag window: {np.percentile(latency_measurements, 95):.4f} ms")
    
    # Min-Max Normalize scores per tag based on Normal data distribution
    norm_pasad_scaled = {}
    norm_epasad_scaled = {}
    att_pasad_scaled = {}
    att_epasad_scaled = {}
    
    tag_thresholds_pasad = {}
    tag_thresholds_epasad = {}
    
    for tag in cont_cols:
        # PASAD
        min_p = np.min(norm_pasad_raw[tag])
        max_p = np.max(norm_pasad_raw[tag])
        range_p = max_p - min_p if max_p > min_p else 1.0
        
        norm_pasad_scaled[tag] = (norm_pasad_raw[tag] - min_p) / range_p
        att_pasad_scaled[tag] = (att_pasad_raw[tag] - min_p) / range_p
        tag_thresholds_pasad[tag] = np.percentile(norm_pasad_scaled[tag], THRESHOLD_PERCENTILE)
        
        # EPASAD
        min_e = np.min(norm_epasad_raw[tag])
        max_e = np.max(norm_epasad_raw[tag])
        range_e = max_e - min_e if max_e > min_e else 1.0
        
        norm_epasad_scaled[tag] = (norm_epasad_raw[tag] - min_e) / range_e
        att_epasad_scaled[tag] = (att_epasad_raw[tag] - min_e) / range_e
        tag_thresholds_epasad[tag] = np.percentile(norm_epasad_scaled[tag], THRESHOLD_PERCENTILE)
        
        tag_models[tag]['threshold_pasad'] = tag_thresholds_pasad[tag]
        tag_models[tag]['threshold_epasad'] = tag_thresholds_epasad[tag]
        
    # FUSION
    fused_norm_pasad = fuse_tag_scores(norm_pasad_scaled, method='max')
    fused_att_pasad = fuse_tag_scores(att_pasad_scaled, method='max')
    
    fused_norm_epasad = fuse_tag_scores(norm_epasad_scaled, method='max')
    fused_att_epasad = fuse_tag_scores(att_epasad_scaled, method='max')
    
    # GLOBAL THRESHOLD (Derived from Normal Fused Scores)
    thresh_pasad = compute_threshold(fused_norm_pasad, method='percentile')
    thresh_epasad = compute_threshold(fused_norm_epasad, method='percentile')
    
    # Alignment: Ground Truth from Attack set
    # Note: sliding window reduces length by L-1
    # We take the ground truth corresponding to the END of the window (i.e. i+L-1)
    L_default = DEFAULT_L
    gt_indices = [s + L_default - 1 for s in att_starts]
    y_true_att = df_att['attack'].values[gt_indices]
    
    print("\n--- PASAD Evaluation ---")
    res_pasad = evaluate(y_true_att, fused_att_pasad, thresh_pasad, att_pasad_scaled, tag_thresholds_pasad)
    res_pasad['Variant'] = 'PASAD'
    res_pasad['Stage'] = stg
    
    print("\n--- EPASAD Evaluation ---")
    res_epasad = evaluate(y_true_att, fused_att_epasad, thresh_epasad, att_epasad_scaled, tag_thresholds_epasad)
    res_epasad['Variant'] = 'EPASAD'
    res_epasad['Stage'] = stg
    
    results.extend([res_pasad, res_epasad])
    
    # Save Model Artifacts
    joblib.dump(tag_models, f"{models_dir}/pasad_stage{stg}.pkl")
    print(f"Saved model artifacts to {models_dir}/pasad_stage{stg}.pkl")

# Final Results
df_results = pd.DataFrame(results)
display(df_results)
